[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IyadSultan/CCI/blob/main/session5/solutions/Lab3_LangGraph_Workflows_Solutions.ipynb)

# Lab 3: LangGraph — Stateful Workflows — SOLUTIONS
## CCI Session 5

**Duration:** 15 minutes

### Clinical Scenario
> Build a clinical triage system: a patient presents with symptoms, the system classifies severity (low/medium/high), routes to appropriate handling (self-care advice, schedule appointment, or emergency alert), and produces a structured output.

### Objective
- Build a `StateGraph` with typed state
- Create nodes that read and update state
- Add conditional routing based on state
- Visualize the workflow graph

---
## Setup

In [ ]:
!pip install -q langchain langchain-openai langgraph grandalf

In [ ]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print("Setup complete.")

---
## Cell 1 — Define the State Schema

In LangGraph, the **state** is a typed dictionary that flows through the graph. Each node can read from and write to the state.

Think of it as the patient's chart that gets passed from station to station in the triage process.

In [ ]:
from typing import TypedDict, Literal, Annotated
from langgraph.graph import add_messages

# SOLUTION: Define the TriageState TypedDict
class TriageState(TypedDict):
    patient_symptoms: str
    severity: str
    recommendation: str
    messages: Annotated[list, add_messages]

print("State schema defined.")
print("Fields:", list(TriageState.__annotations__.keys()))

---
## Cell 2 — Define the Nodes

Each node is a function that:
1. Receives the current state
2. Does some work (may call an LLM)
3. Returns a dict with the state keys it wants to update

Our triage workflow has 5 nodes:
- **intake_node**: Records the patient's symptoms
- **classify_node**: Uses LLM to classify severity
- **low_severity_node**: Provides self-care advice
- **medium_severity_node**: Schedules an appointment
- **high_severity_node**: Triggers emergency alert

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage
from pydantic import BaseModel, Field

# Node 1: Intake — records symptoms into state
def intake_node(state: TriageState) -> dict:
    """Process patient intake — record symptoms."""
    print("[INTAKE] Processing patient symptoms...")
    return {
        "messages": [
            SystemMessage(content="You are a clinical triage nurse at KHCC."),
            HumanMessage(content=f"Patient presents with: {state['patient_symptoms']}")
        ]
    }

# Node 2: Classify — uses LLM to determine severity
class SeverityClassification(BaseModel):
    """Classification of patient symptom severity."""
    severity: Literal["low", "medium", "high"] = Field(
        description="Severity level: low=self-care, medium=appointment needed, high=emergency"
    )
    reasoning: str = Field(description="Brief clinical reasoning for the classification")

# SOLUTION: classify_node
def classify_node(state: TriageState) -> dict:
    """Classify symptom severity using LLM."""
    print("[CLASSIFY] Analyzing symptom severity...")
    classifier = llm.with_structured_output(SeverityClassification)
    result = classifier.invoke(
        f"Classify the severity of these symptoms for triage purposes. "
        f"low = can self-manage at home, medium = needs clinic appointment within days, "
        f"high = needs immediate emergency attention.\n\n"
        f"Patient symptoms: {state['patient_symptoms']}"
    )
    print(f"  -> Classified as: {result.severity} ({result.reasoning})")
    return {"severity": result.severity}

# Node 3: Low severity — self-care advice
def low_severity_node(state: TriageState) -> dict:
    """Handle low severity — provide self-care advice."""
    print("[LOW] Generating self-care advice...")
    response = llm.invoke(
        f"As a triage nurse, provide brief self-care advice for: {state['patient_symptoms']}. "
        f"Keep it to 2-3 sentences. Mention when to seek further care."
    )
    return {"recommendation": f"SELF-CARE: {response.content}"}

# SOLUTION: Node 4: Medium severity — schedule appointment
def medium_severity_node(state: TriageState) -> dict:
    """Handle medium severity — schedule appointment."""
    print("[MEDIUM] Scheduling appointment...")
    response = llm.invoke(
        f"As a triage nurse, recommend an appropriate clinic appointment for: {state['patient_symptoms']}. "
        f"Specify which specialty to see, urgency (within how many days), and any tests to order beforehand. "
        f"Keep it to 3-4 sentences."
    )
    return {"recommendation": f"APPOINTMENT: {response.content}"}

# SOLUTION: Node 5: High severity — emergency alert
def high_severity_node(state: TriageState) -> dict:
    """Handle high severity — emergency alert."""
    print("[HIGH] EMERGENCY ALERT!")
    response = llm.invoke(
        f"As an emergency triage nurse, provide immediate instructions for: {state['patient_symptoms']}. "
        f"Include: immediate actions, what NOT to do, and what to tell the emergency team. "
        f"Be direct and concise."
    )
    return {"recommendation": f"EMERGENCY: {response.content}"}

print("All nodes defined.")

---
## Cell 3 — Define Conditional Edges

After classification, the graph needs to route to the correct handler based on severity. This is done with a **conditional edge**.

In [ ]:
# SOLUTION: Define the routing function
def route_by_severity(state: TriageState) -> str:
    """Route to the appropriate handler based on severity."""
    severity = state["severity"]
    if severity == "low":
        return "low_handler"
    elif severity == "medium":
        return "medium_handler"
    elif severity == "high":
        return "high_handler"
    else:
        return "medium_handler"  # Default to medium if unclear

print("Routing function defined.")

---
## Cell 4 — Build and Compile the Graph

Now we assemble the nodes and edges into a `StateGraph`.

In [ ]:
from langgraph.graph import StateGraph, START, END

# SOLUTION: Build the graph
graph = StateGraph(TriageState)

# Add nodes
graph.add_node("intake", intake_node)
graph.add_node("classify", classify_node)
graph.add_node("low_handler", low_severity_node)
graph.add_node("medium_handler", medium_severity_node)
graph.add_node("high_handler", high_severity_node)

# Add edges
graph.add_edge(START, "intake")
graph.add_edge("intake", "classify")
graph.add_conditional_edges("classify", route_by_severity)
graph.add_edge("low_handler", END)
graph.add_edge("medium_handler", END)
graph.add_edge("high_handler", END)

# Compile
triage_app = graph.compile()

print("Graph compiled successfully!")

---
## Cell 5 — Run with Different Patient Scenarios

Let's test our triage system with patients of different severity levels.

In [ ]:
# Test scenarios
scenarios = [
    {
        "name": "Scenario 1: Mild symptoms",
        "symptoms": "Mild headache for 2 days, no fever, no vision changes. Patient is otherwise healthy."
    },
    {
        "name": "Scenario 2: Moderate symptoms",
        "symptoms": "Persistent cough for 2 weeks, low-grade fever (37.8C), unintentional weight loss of 3kg. Former smoker."
    },
    {
        "name": "Scenario 3: Severe symptoms",
        "symptoms": "Sudden severe chest pain, difficulty breathing, sweating, pain radiating to left arm. History of hypertension."
    }
]

# SOLUTION: Run each scenario
for scenario in scenarios:
    print("\n" + "=" * 60)
    print(f"  {scenario['name']}")
    print("=" * 60)
    print(f"Symptoms: {scenario['symptoms'][:80]}...")
    print()

    result = triage_app.invoke({
        "patient_symptoms": scenario["symptoms"],
        "severity": "",
        "recommendation": "",
        "messages": []
    })

    print(f"\nSeverity: {result['severity'].upper()}")
    print(f"Recommendation: {result['recommendation'][:300]}")

---
## Cell 6 — Visualize the Graph

LangGraph can generate a visual representation of your workflow.

In [ ]:
# Visualize the graph as ASCII
print("Graph structure:")
print()
triage_app.get_graph().print_ascii()

# You can also get a Mermaid diagram for rendering
print("\n\nMermaid diagram (paste into mermaid.live):")
print(triage_app.get_graph().draw_mermaid())

---
## Stretch Challenge — Memory with MemorySaver

In [ ]:
# STRETCH SOLUTION: Add memory to the triage system
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()
triage_app_with_memory = graph.compile(checkpointer=memory)

# First visit
config = {"configurable": {"thread_id": "patient-MRN004"}}

result1 = triage_app_with_memory.invoke(
    {
        "patient_symptoms": "Mild nausea after chemotherapy, able to eat small meals. No vomiting.",
        "severity": "",
        "recommendation": "",
        "messages": []
    },
    config=config
)

print("FIRST VISIT")
print(f"Severity: {result1['severity']}")
print(f"Recommendation: {result1['recommendation'][:200]}")

print("\n" + "=" * 60)

# Second visit — same patient, worse symptoms
result2 = triage_app_with_memory.invoke(
    {
        "patient_symptoms": "Severe nausea and vomiting for 24 hours, unable to keep fluids down. Dizziness when standing. Following up from mild nausea yesterday.",
        "severity": "",
        "recommendation": "",
        "messages": []
    },
    config=config
)

print("\nSECOND VISIT (same patient, worsening)")
print(f"Severity: {result2['severity']}")
print(f"Recommendation: {result2['recommendation'][:200]}")

### KHCC Connection
This triage workflow pattern is directly applicable at KHCC:
- Emergency department triage for oncology patients
- Phone triage for patients calling with symptoms between treatments
- Chemotherapy side-effect severity assessment
- Routing patients to the right clinic (medical oncology, radiation, surgical, palliative)
- The graph can be extended with more nodes for lab ordering, specialist referral, etc.